In [8]:
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np

# Load the dataset and preprocess the images
dataset_path = 'dataset'
batch_size = 32
image_size = (224, 224)

# Specify class names
class_names = ['benign', 'malignant']

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=image_size,
    batch_size=batch_size
)
validation_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=image_size,
    batch_size=batch_size
)
validation_dataset = validation_dataset.map(lambda x, y: (tf.image.per_image_standardization(x), y))

# Add data augmentation
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.experimental.preprocessing.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.experimental.preprocessing.RandomRotation(0.2),
    tf.keras.layers.experimental.preprocessing.RandomZoom(0.2),
    tf.keras.layers.RandomBrightness(factor=0.2),
    tf.keras.layers.experimental.preprocessing.RandomContrast(factor=0.2),
    tf.keras.layers.experimental.preprocessing.RandomTranslation(height_factor=0.1, width_factor=0.1)
    
])



# Add data augmentation to the validation dataset
augmented_validation_dataset = validation_dataset.map(lambda x, y: (data_augmentation(x), y))
normalized_and_augmented_dataset = train_dataset.map(lambda x, y: (data_augmentation(x), y))


# Create and compile the model
model = tf.keras.Sequential([
    tf.keras.layers.experimental.preprocessing.Rescaling(1./255, input_shape=(image_size[0], image_size[1], 3)),
    tf.keras.layers.Conv2D(16, 3, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),  # Add dropout for regularization
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.4),  # Add dropout for regularization
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),  # Add dropout for regularization
    tf.keras.layers.Dense(1, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Implement Early Stopping
#early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Train the model
history = model.fit(
    normalized_and_augmented_dataset,
    epochs=10,
    validation_data=augmented_validation_dataset,  # Provide validation data during training
    #callbacks=[early_stopping]
)

# Evaluate the model on the test dataset
test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset='validation',
    seed=72,
    image_size=image_size,
    batch_size=batch_size
)

test_dataset = test_dataset.map(lambda x, y: (tf.image.per_image_standardization(x), y))

loss, accuracy = model.evaluate(test_dataset)
print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

# Make predictions on the test dataset
test_images = []
test_labels = []
for images, labels in test_dataset:
    test_images.extend(images.numpy())
    test_labels.extend(labels.numpy())

test_images = np.array(test_images)
test_labels = np.array(test_labels)

predictions = model.predict(test_images)
predicted_labels = np.argmax(predictions, axis=1)

# Calculate evaluation metrics
accuracy = accuracy_score(test_labels, predicted_labels)
precision = precision_score(test_labels, predicted_labels)
recall = recall_score(test_labels, predicted_labels)
f1 = f1_score(test_labels, predicted_labels)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

# Confusion Matrix
conf_matrix = confusion_matrix(test_labels, predicted_labels)
print("Confusion Matrix:\n", conf_matrix)

# Plot training and validation loss curves
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss') 
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()


# Create a grid of images with their predicted labels
fig, axes = plt.subplots(6, 6, figsize=(16, 16))
axes = axes.ravel()

predicted_labels = predicted_labels.flatten()  # Add this line

for i in range(36):
    axes[i].imshow(test_images[i])
    axes[i].set_title("Predicted: {}".format(class_names[predicted_labels[i]]))
    axes[i].axis('off')

plt.tight_layout()
plt.show()



Found 3297 files belonging to 2 classes.
Using 2638 files for training.
Found 3297 files belonging to 2 classes.
Using 659 files for validation.
Epoch 1/10
83/83 [==============================] - 45s 508ms/step - loss: 0.0000e+00 - accuracy: 0.7949 - val_loss: 0.0000e+00 - val_accuracy: 0.8194
Epoch 2/10
24/83 [=======>......................] - ETA: 24s - loss: 0.0000e+00 - accuracy: 0.7747

KeyboardInterrupt: 